# System 2 — RAG Fact-Verification Engine

Arabic fake news detection for Egyptian dialect.

- **Bucket A** — Known fake claims (AraBERT cosine similarity)
- **Bucket B** — Truth KB propositions + AraFacts descriptions (Hybrid BM25 + AraBERT)
- **Normalization** — Egyptian dialect (CAI) → MSA via EGY_TO_MSA dictionary


## 1. Setup — models, data, and helper functions
Run this whole section once, top to bottom. Loads all models (cached locally, fast after first run) and connects to the Chroma vector DB (Bucket A + B already built on disk — these cells detect that and skip re-ingestion).

In [ ]:
import os, re, subprocess
import pandas as pd
import numpy as np
import torch
import chromadb
from transformers import pipeline, AutoTokenizer, AutoModel
from camel_tools.utils.normalize import normalize_alef_ar, normalize_unicode

# Install rank-bm25 if not present
subprocess.run(["pip", "install", "rank-bm25", "-q"])
from rank_bm25 import BM25Okapi

# ── Paths ────────────────────────────────────────────────────────────────────
BASE   = "/Users/russelltamer/Desktop/system 2 RAG"
PROPS  = f"{BASE}/df_propositions_all.csv"      # merged: original 8,376 + v2 5,808 + SANAD 6,656 = 20,687
CHROMA = f"{BASE}/chroma_db"

for name, path in [("Propositions CSV", PROPS)]:
    status = "✅" if os.path.exists(path) else "❌ MISSING"
    print(f"{status}  {name}: {path}")

print("\n✅ All imports OK")

In [ ]:
# ── Dialect classifier (CAMeL) ───────────────────────────────────────────────
print("Loading dialect classifier...")
dialect_classifier = pipeline(
    "text-classification",
    model="CAMeL-Lab/bert-base-arabic-camelbert-mix-did-madar-corpus6"
)
print("✅ Dialect classifier loaded  (label 'CAI' = Egyptian Arabic)")

# ── multilingual-e5-large ─────────────────────────────────────────────────────
# Retrieval-optimized: trained on (query, passage) pairs.
# Requires prefix: "query: " for queries, "passage: " for documents.
# v1 used AraBERT (CLS token, 768-dim) — task mismatch caused P@1=0.415.
# E5-large is purpose-built for retrieval → much stronger semantic matching.
print("\nLoading multilingual-e5-large...")
E5_MODEL_NAME = "intfloat/multilingual-e5-large"
e5_tokenizer  = AutoTokenizer.from_pretrained(E5_MODEL_NAME)
e5_model      = AutoModel.from_pretrained(E5_MODEL_NAME)
e5_model.eval()
print("✅ E5-large loaded  (1024-dim, mean pooling)")

def _mean_pool(token_embeds, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(token_embeds.size()).float()
    return (token_embeds * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

def get_embedding(text: str, prefix: str = "query") -> np.ndarray:
    inp = e5_tokenizer(f"{prefix}: {text}", return_tensors="pt",
                       truncation=True, max_length=512, padding=True)
    with torch.no_grad():
        out = e5_model(**inp)
    vec = _mean_pool(out.last_hidden_state, inp['attention_mask'])
    vec = torch.nn.functional.normalize(vec, p=2, dim=1)
    return vec.squeeze().numpy()

def get_embeddings_batch(texts: list, batch_size: int = 16,
                          prefix: str = "passage") -> np.ndarray:
    prefixed = [f"{prefix}: {t}" for t in texts]
    all_vecs = []
    for i in range(0, len(prefixed), batch_size):
        batch = prefixed[i:i+batch_size]
        inp = e5_tokenizer(batch, return_tensors="pt", truncation=True,
                           max_length=512, padding=True)
        with torch.no_grad():
            out = e5_model(**inp)
        vecs = _mean_pool(out.last_hidden_state, inp['attention_mask'])
        vecs = torch.nn.functional.normalize(vecs, p=2, dim=1)
        all_vecs.append(vecs.numpy())
        if (i // batch_size) % 10 == 0:
            print(f"  Embedded {i+len(batch)}/{len(texts)}")
    return np.vstack(all_vecs)

vec = get_embedding("اختبار")
print(f"✅ Embedding shape: {vec.shape}")  # expect (1024,)

# ── NER model ─────────────────────────────────────────────────────────────────
print("\nLoading NER model...")
ner_pipeline_ar = pipeline(
    "ner",
    model="CAMeL-Lab/bert-base-arabic-camelbert-mix-ner",
    aggregation_strategy="simple",
    device=-1
)
print("✅ NER model loaded")

In [ ]:
# ── AraFacts 2 ───────────────────────────────────────────────────────────────
df_arafacts = pd.read_csv(f"{BASE}/AraFacts 2.csv")
print(f"AraFacts shape: {df_arafacts.shape}")
print(f"Columns: {df_arafacts.columns.tolist()}")
print(df_arafacts['normalized_label'].value_counts())

# Filter descriptions for Bucket B (non-empty, ≥ 40 chars)
df_desc = df_arafacts[
    df_arafacts['description'].notna() &
    (df_arafacts['description'].str.len() > 40)
].copy().reset_index(drop=True)
print(f"\nAraFacts descriptions available for Bucket B: {len(df_desc)}")


**Claude API setup** — needed for decontextualizing the top Bucket B evidence shown to users. Must run *before* the main pipeline function below (this was previously positioned after it by mistake, which silently disabled this feature).

In [ ]:
from dotenv import load_dotenv
load_dotenv("/Users/russelltamer/Desktop/system 2 RAG/.env")
import os
assert os.environ.get("ANTHROPIC_API_KEY"), "Key not loaded — check .env path"
print("✅ API key loaded safely")


In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "anthropic"])
import anthropic
import os

In [ ]:
claude_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

def decontextualize(title, proposition):
    prompt = f"""العنوان: {title}
الجملة المستخرجة من المقال: {proposition}

أعد كتابة الجملة المستخرجة بحيث تكون مستقلة ومفهومة بذاتها، بإضافة السياق الناقص (الموضوع، الفاعل) من العنوان فقط دون إضافة معلومات غير موجودة. أعد فقط الجملة المعاد كتابتها بدون أي شرح إضافي."""
    resp = claude_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text.strip()

**Bucket B** — live news knowledge base (28,800+ propositions).

In [ ]:
client = chromadb.PersistentClient(path=CHROMA)

# ── Guards ───────────────────────────────────────────────────────────────────
# Bucket B = scraped news propositions ONLY (independent from Bucket A)
# Sources: original KB (8,376) + scraped v2 (5,808) + SANAD (6,503) = 20,687 total
# AraFacts descriptions removed — they overlap with Bucket A's AraFacts claims
EXPECTED_B_MIN = 20000

try:
    col_b = client.get_collection("bucket_b_propositions")
    if col_b.count() >= EXPECTED_B_MIN:
        print(f"✅ Bucket B already ingested: {col_b.count()} docs — skipping")
    else:
        raise ValueError(f"Incomplete ({col_b.count()} < {EXPECTED_B_MIN})")
except Exception as e:
    print(f"Re-ingesting Bucket B ({e})...")
    try: client.delete_collection("bucket_b_propositions")
    except: pass
    col_b = client.create_collection(
        "bucket_b_propositions", metadata={"hnsw:space": "cosine"})

    # ── Clean propositions from merged news KB (independent evidence source) ─
    df_props = pd.read_csv(PROPS)
    df_props = df_props.drop_duplicates(subset='proposition').reset_index(drop=True)
    print(f"Propositions loaded: {len(df_props)}")
    print(f"Sources: {df_props['source'].value_counts().to_dict()}")

    print("\nEmbedding propositions...")
    vecs_props = get_embeddings_batch(df_props['proposition'].tolist(), batch_size=32)

    BATCH = 500
    for i in range(0, len(df_props), BATCH):
        b_df  = df_props.iloc[i:i+BATCH]
        b_vec = vecs_props[i:i+BATCH]
        col_b.add(
            ids=[f"prop_{j}" for j in range(i, i+len(b_df))],
            embeddings=b_vec.tolist(),
            documents=b_df['proposition'].tolist(),
            metadatas=[{
                "article_id": str(r['article_id']),
                "title":      str(r['title']),
                "source":     str(r['source']),
            } for _, r in b_df.iterrows()]
        )

    print(f"\n✅ Bucket B complete: {col_b.count()} propositions")
    print(f"   Sources: {df_props['source'].value_counts().head(5).to_dict()}")
    print(f"   Note: AraFacts descriptions removed — Bucket B is now independent from Bucket A")

**Bucket A** — verified claims database (AraFacts + Saheeh Masr).

In [ ]:
# ── Bucket A: ALL verified claims (TRUE + FALSE + Partly-False) ──────────────
# Renamed collection to reflect correct scope
EXPECTED_A_MIN = 21000   # AraFacts all claims (~18,598) + Saheeh Masr all (~3,019)

LABEL_MAP = {
    "FALSE":        "FALSE",
    "Partly-false": "PARTLY_FALSE",
    "Partly-False": "PARTLY_FALSE",
    "TRUE":         "TRUE",
    "Sarcasm":      "SARCASM",
    "Unverifiable": "UNVERIFIABLE",
}

try:
    col_a = client.get_collection("bucket_a_verified_claims")
    if col_a.count() >= EXPECTED_A_MIN:
        print(f"✅ Bucket A already ingested: {col_a.count()} docs — skipping")
    else:
        raise ValueError(f"Incomplete ({col_a.count()} < {EXPECTED_A_MIN})")
except Exception as e:
    print(f"Re-ingesting Bucket A ({e})...")

    # Drop old collections (both old name and new name if they exist)
    for name in ["bucket_a_fake_claims", "bucket_a_verified_claims"]:
        try: client.delete_collection(name)
        except: pass

    col_a = client.create_collection(
        "bucket_a_verified_claims", metadata={"hnsw:space": "cosine"})

    # ── AraFacts: all 18,598 claims ───────────────────────────────────────────
    df_all_arafacts = df_arafacts[['claim','normalized_label','source','description','date']].copy()
    df_all_arafacts['label'] = df_all_arafacts['normalized_label'].map(LABEL_MAP).fillna("UNKNOWN")
    df_all_arafacts = df_all_arafacts.dropna(subset=['claim']).reset_index(drop=True)
    print(f"AraFacts claims: {len(df_all_arafacts)}")
    print(df_all_arafacts['label'].value_counts().to_string())

    print("\nEmbedding AraFacts claims...")
    vecs_arafacts = get_embeddings_batch(df_all_arafacts['claim'].tolist(), batch_size=32)

    BATCH = 500
    for i in range(0, len(df_all_arafacts), BATCH):
        b_df  = df_all_arafacts.iloc[i:i+BATCH]
        b_vec = vecs_arafacts[i:i+BATCH]
        col_a.add(
            ids=[f"ara_{j}" for j in range(i, i+len(b_df))],
            embeddings=b_vec.tolist(),
            documents=b_df['claim'].tolist(),
            metadatas=[{
                "label":       str(r['label']),
                "source":      str(r['source']),
                "description": str(r['description'])[:400] if pd.notna(r['description']) else "",
                "date":        str(r['date']) if pd.notna(r['date']) else "",
                "dialect":     "",
            } for _, r in b_df.iterrows()]
        )
    print(f"AraFacts ingested: {col_a.count()}")

    # ── Saheeh Masr: all 3,019 claims (TRUE + FALSE) ─────────────────────────
    df_saheeh = pd.read_csv(f"{BASE}/saheeh_masr_claims.csv")
    df_saheeh['claim'] = df_saheeh['claim'].str.replace(
        r'\n?اقرأ التصحيح.*$', '', regex=True).str.strip()
    df_saheeh = df_saheeh.drop_duplicates(subset='claim').reset_index(drop=True)
    df_saheeh = df_saheeh[df_saheeh['claim'].str.len() >= 15].reset_index(drop=True)
    df_saheeh['label'] = df_saheeh['verdict'].map({True: "TRUE", False: "FALSE"})
    print(f"\nSaheeh Masr claims: {len(df_saheeh)}")
    print(df_saheeh['label'].value_counts().to_string())

    print("\nEmbedding Saheeh Masr claims...")
    vecs_saheeh = get_embeddings_batch(df_saheeh['claim'].tolist(), batch_size=32)

    start_id = col_a.count()
    for i in range(0, len(df_saheeh), BATCH):
        b_df  = df_saheeh.iloc[i:i+BATCH]
        b_vec = vecs_saheeh[i:i+BATCH]
        col_a.add(
            ids=[f"saheeh_{start_id + j}" for j in range(i, i+len(b_df))],
            embeddings=b_vec.tolist(),
            documents=b_df['claim'].tolist(),
            metadatas=[{
                "label":       str(r['label']),
                "source":      "saheeh_masr",
                "description": "",
                "date":        str(r['date']) if pd.notna(r['date']) else "",
                "dialect":     "EGY",
            } for _, r in b_df.iterrows()]
        )

    print(f"\n✅ Bucket A complete: {col_a.count()} total verified claims")
    print(f"   Sources: AraFacts ({len(df_all_arafacts)}) + Saheeh Masr ({len(df_saheeh)})")


**Egyptian dialect → MSA normalization**, used to clean queries before retrieval.

In [ ]:
# Egyptian dialect (CAI) → MSA word-level mapping
EGY_TO_MSA = {
    # Pronouns
    "انا": "أنا",   "احنا": "نحن",   "انت": "أنت",   "انتو": "أنتم",
    # Negation / existence
    "مش": "ليس",    "مفيش": "لا يوجد",  "معندوش": "ليس لديه",
    # Conjunctions
    "لو": "إذا",    "عشان": "لأن",    "علشان": "لأن",  "بس": "لكن",
    # Adverbs
    "كده": "هكذا",  "كدا": "هكذا",    "اوي": "جداً",
    "دلوقتي": "الآن", "دلوقت": "الآن", "امبارح": "أمس",
    # Interrogatives
    "ايه": "ماذا",  "فين": "أين",     "ازاي": "كيف",
    "امتى": "متى",  "ليه": "لماذا",   "مين": "من",
    # Common verbs (active)
    "جه": "جاء",    "راح": "ذهب",     "قعد": "جلس",    "شاف": "رأى",
    "بيقول": "يقول", "بيعمل": "يفعل", "بيجي": "يأتي",  "بيروح": "يذهب",
    "بيحصل": "يحدث", "بيصير": "يحدث", "بيحدث": "يحدث",
    "بيتكلم": "يتحدث", "بيشتغل": "يعمل", "بيكذب": "يكذب",
    "بيقدر": "يستطيع", "بيحاول": "يحاول",
    "بيشوف": "يرى",  "بنشوف": "نرى",
    "بنقول": "نقول", "بنعمل": "نفعل",
    "هيعمل": "سيفعل", "هيجي": "سيأتي", "هيروح": "سيذهب",
    "عارف": "يعرف",  "عارفة": "تعرف",
    # Passive verbs
    "اتعمل": "صُنع", "اتقال": "قيل",  "اتكلم": "تحدث",
    "اتحط": "وُضع",  "اتبنى": "بُني",
    # Demonstratives (also handled by word-order fix below)
    "ده": "هذا",    "دي": "هذه",     "دول": "هؤلاء",
    # Relative pronoun
    "اللي": "الذي",
    # Misc
    "كمان": "أيضاً", "الاول": "أولاً", "تاني": "ثانياً",
    "كدب": "كذب", "حاجة": "شيء",
}

# ── POS-aware demonstrative reordering (upgrade over position-only heuristic) ──
from camel_tools.disambig.bert import BERTUnfactoredDisambiguator
print("Loading Egyptian Arabic morphological disambiguator...")
_egy_disambiguator = BERTUnfactoredDisambiguator.pretrained(model_name='egy')
print("✅ Disambiguator loaded")

def fix_demonstrative_order(text: str) -> str:
    """
    Move demonstrative (هذا/هذه/هؤلاء) to precede the noun it modifies, using real POS
    tagging instead of a position-only heuristic. Only reorders when the word immediately
    before the demonstrative is actually tagged as a noun -- e.g. correctly leaves
    "تغير مناخي هذا" unchanged since "مناخي" is an adjective, not a noun, avoiding the
    nonsense "هذا مناخي" the old position-only heuristic produced.
    """
    words = text.split()
    demonstratives = {"هذا", "هذه", "هؤلاء"}
    demo_indices = [i for i, w in enumerate(words) if w in demonstratives]
    if not demo_indices:
        return text

    analyses = _egy_disambiguator.disambiguate(words)
    pos_tags = [a.analyses[0].analysis.get('pos') if a.analyses else None for a in analyses]

    result = words[:]
    for idx in demo_indices:
        if idx == 0:
            continue
        if pos_tags[idx - 1] == "noun":
            result[idx - 1], result[idx] = result[idx], result[idx - 1]
        # else: leave it -- not a noun, don't guess
    return " ".join(result)

def normalize_to_msa(text: str) -> str:
    text = normalize_unicode(text)
    text = normalize_alef_ar(text)          # ألف-only — safe; NOT alef_maksura/teh_marbuta
    words = text.split()
    text  = " ".join(EGY_TO_MSA.get(w, w) for w in words)
    text  = fix_demonstrative_order(text)
    return text

def detect_dialect(text: str):
    result = dialect_classifier(text[:512])[0]
    return result['label'], round(result['score'], 3)

def prepare_query(claim: str, verbose: bool = True) -> str:
    """Detect dialect; normalize EGY (CAI) → MSA before embedding."""
    dialect, conf = detect_dialect(claim)
    if verbose:
        print(f"  Dialect: {dialect} (conf={conf})")
    if dialect == 'CAI' and conf > 0.5:
        normalized = normalize_to_msa(claim)
        if verbose:
            print(f"  Original:   {claim}")
            print(f"  Normalized: {normalized}")
        return normalized
    return claim

# Quick smoke test
print(prepare_query("مفيش دليل على الكلام ده اوي"))


**Arabic stemming + stopwords** — used by BM25 tokenization and the lexical-overlap safety check.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "nltk", "-q"])
import nltk
nltk.download('punkt', quiet=True)
from nltk.stem.isri import ISRIStemmer

stemmer = ISRIStemmer()

# Arabic stopwords
ARABIC_STOPWORDS = {
    "من","في","على","إلى","عن","مع","هذا","هذه","هو","هي","هم","هن",
    "أن","إن","كان","كانت","لا","لم","لن","قد","قال","قالت",
    "ما","ماذا","التي","الذي","الذين","وهو","وهي","وكان","ومن",
    "يوجد","توجد","وجود","غير","بعد","قبل","عند","حتى","إذا",
    "لكن","أو","بل","ثم","حيث","كيف","لماذا","متى","أين",
    "كل","بعض","جميع","أي","كما","أيضا","فقط","جدا","وقد",
    "وفي","وعلى","وإلى","وأن","وكانت","وكان",
    "لا","على","عن","في","من","إن","أن","يا","هل","نعم",
}

def bm25_tokenize(text: str) -> list:
    """Stopword removal + ISRI stemming for Arabic BM25 (following Paper 3)."""
    tokens = [w for w in text.split() if w not in ARABIC_STOPWORDS and len(w) > 2]
    return [stemmer.stem(t) for t in tokens]

# ── Bucket B BM25 index ───────────────────────────────────────────────────────
df_all_b = pd.read_csv(PROPS)
df_all_b = df_all_b.drop_duplicates(subset='proposition').reset_index(drop=True)
df_all_b = df_all_b.rename(columns={'proposition': 'text'})
print(f"Bucket B docs: {len(df_all_b)}")

print("Building BM25 index…")
tokenized_all = [bm25_tokenize(doc) for doc in df_all_b['text'].tolist()]
# k1=1.2, b=0.75 calibrated for Arabic news text (Paper 4 — Hybrid RAG for Islamic QA)
bm25 = BM25Okapi(tokenized_all, k1=1.2, b=0.75)
print(f"✅ BM25 index ready  (k1=1.2, b=0.75, ISRI stemming, {len(df_all_b)} docs)")

**Named entity extraction** — boosts retrieval ranking when a claim and evidence share a named entity (person/place/org).

In [ ]:
def extract_entities(text: str) -> set:
    """Extract named entities from Arabic text for boosting."""
    try:
        results = ner_pipeline_ar(text[:512])
        return {r['word'].strip() for r in results
                if r['score'] > 0.7 and len(r['word'].strip()) > 2}
    except Exception:
        return set()

def hybrid_search(query_text: str, k: int = 5, k_rrf: int = 60) -> list:
    """
    RRF hybrid retrieval: 1/(k_rrf + rank_e5) + 1/(k_rrf + rank_bm25)
    + NER entity boosting.
    Upgraded from v1 weighted sum (alpha*arabert + (1-alpha)*bm25):
      - RRF uses ranks not raw scores → no normalization needed, no alpha to tune
      - E5-large vectors replace AraBERT → better retrieval alignment
      - NER boost fixes geographic/person entity conflation
    """
    # BM25 ranking
    tokens     = bm25_tokenize(query_text)
    bm25_sc    = bm25.get_scores(tokens)
    bm25_order = np.argsort(bm25_sc)[::-1]
    bm25_ranks = {int(idx): rank + 1 for rank, idx in enumerate(bm25_order)}

    # E5-large ranking from ChromaDB
    qvec    = get_embedding(query_text, prefix="query")
    results = col_b.query(query_embeddings=[qvec.tolist()], n_results=50)
    e5_ranks = {}
    for rank, doc_id in enumerate(results['ids'][0]):
        idx = int(doc_id.split('_')[1])   # "prop_1234" → 1234
        e5_ranks[idx] = rank + 1

    # NER entities from claim for entity overlap boosting
    claim_entities = extract_entities(query_text)

    # Candidate pool: top-50 BM25 ∪ top-50 E5
    all_idx = set(int(i) for i in bm25_order[:50]) | set(e5_ranks.keys())

    candidates = []
    for idx in all_idx:
        r_bm25 = bm25_ranks.get(idx, 100)
        r_e5   = e5_ranks.get(idx, 100)
        rrf    = 1 / (k_rrf + r_bm25) + 1 / (k_rrf + r_e5)

        prop_text = df_all_b.iloc[idx]['text']
        if claim_entities:
            overlap = sum(1 for e in claim_entities if e in prop_text)
            rrf += overlap * 0.005   # small boost per matching entity

        candidates.append({
            "proposition":  prop_text,
            "title":        df_all_b.iloc[idx]['title'],
            "source":       df_all_b.iloc[idx].get('source', ''),
            "hybrid_score": round(rrf, 5),
            "bm25_rank":    r_bm25,
            "e5_rank":      r_e5,
        })

    candidates.sort(key=lambda x: x['hybrid_score'], reverse=True)

    seen, hits = set(), []
    for c in candidates:
        t = c['title']
        if t not in seen:
            seen.add(t)
            hits.append(c)
        if len(hits) >= k:
            break

    return hits

print("✅ hybrid_search ready  (RRF fusion + NER entity boosting)")

**Lexical overlap check** — catches cases where two texts look semantically similar (high embedding similarity) but barely share any actual words. Prevents the system from confidently matching a claim to an unrelated stored claim just because they're stylistically alike.

In [ ]:
def lexical_overlap(text1: str, text2: str) -> float:
    """
    Fraction of shared stemmed content words between two texts (Jaccard overlap).
    Used to catch cases where E5 reports high semantic similarity but the two
    texts share almost no actual words — a sign E5 may be matching on style/
    adjectives rather than real subject overlap (e.g. "giant crocodile" vs
    "giant water wheel" — both "giant", completely different subject).
    """
    tokens1 = set(bm25_tokenize(text1))
    tokens2 = set(bm25_tokenize(text2))
    if not tokens1 or not tokens2:
        return 0.0
    return len(tokens1 & tokens2) / len(tokens1 | tokens2)

print("✅ lexical_overlap ready")

**Cross-encoder reranker** — re-scores Bucket B's retrieved candidates for relevance (F1-calibrated threshold: 2.0).

In [ ]:
# ── Cross-encoder Reranker ────────────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "sentence-transformers", "-q"])
from sentence_transformers import CrossEncoder

print("Loading cross-encoder reranker...")
cross_encoder = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")
print("✅ Cross-encoder loaded  (multilingual, trained on MS MARCO)")

def rerank(claim: str, candidates: list, top_k: int = 5) -> list:
    """
    Re-score (claim, proposition) pairs jointly using cross-encoder.
    Unlike cosine similarity (which compares vectors independently),
    the cross-encoder sees both texts together → captures entailment, negation, specificity.
    """
    if not candidates:
        return candidates
    pairs  = [(claim, c['proposition']) for c in candidates]
    scores = cross_encoder.predict(pairs)
    for c, s in zip(candidates, scores):
        c['rerank_score'] = round(float(s), 4)
    reranked = sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)
    return reranked[:top_k]

print("✅ rerank ready")

## 2. Core pipeline — `fact_check_claim()`
This is the main function. Given an Arabic claim, it returns a verdict (TRUE/FALSE/UNVERIFIED), a confidence score, and supporting evidence from both buckets. **This is what System 3 (Youssef) consumes.**

In [ ]:
# Decontextualization of displayed Bucket B evidence (Chen et al. 2023) — applied
# only to the top result shown to a user, not the full index (cost/time constraint).
# Cached to disk + wrapped in try/except so a network/API failure NEVER breaks the
# pipeline — falls back to the original (non-decontextualized) text silently.
import json as _json
import os as _os

DECONTEXT_CACHE_PATH = f"{BASE}/decontext_cache.json"
if _os.path.exists(DECONTEXT_CACHE_PATH):
    with open(DECONTEXT_CACHE_PATH, encoding="utf-8") as _f:
        _decontext_cache = _json.load(_f)
else:
    _decontext_cache = {}

def get_decontextualized(title: str, proposition: str) -> str:
    """Returns decontextualized text if available/cacheable, else the original — never raises."""
    key = title + "||" + proposition
    if key in _decontext_cache:
        return _decontext_cache[key]
    try:
        rewritten = decontextualize(title, proposition)
        _decontext_cache[key] = rewritten
        with open(DECONTEXT_CACHE_PATH, "w", encoding="utf-8") as f:
            _json.dump(_decontext_cache, f, ensure_ascii=False, indent=1)
        return rewritten
    except Exception:
        return proposition  # silent fallback — demo never breaks

VERDICT_TO_FINAL = {
    "HIGH_FAKE_MATCH":        "FALSE",
    "HIGH_TRUE_MATCH":        "TRUE",
    "HIGH_PARTIAL_MATCH":     "UNVERIFIED",
    "POSSIBLE_FAKE":          "FALSE",
    "POSSIBLE_TRUE":          "TRUE",
    "POSSIBLE_MATCH":         "UNVERIFIED",
    "POSSIBLE_PARTIAL_MATCH": "UNVERIFIED",
    "EVIDENCE_FOUND":         "UNVERIFIED",
    "LOW_CONFIDENCE":         "UNVERIFIED",
}

HIGH_TO_POSSIBLE = {
    "HIGH_FAKE_MATCH":    "POSSIBLE_FAKE",
    "HIGH_TRUE_MATCH":    "POSSIBLE_TRUE",
    "HIGH_PARTIAL_MATCH": "POSSIBLE_MATCH",
}

def compute_confidence(signal: str, top_a: float = 0.0, top_rerank: float = None) -> float:
    if signal in ("HIGH_FAKE_MATCH", "HIGH_TRUE_MATCH"):
        return round(min(0.97, top_a), 2)
    elif signal == "HIGH_PARTIAL_MATCH":
        return 0.75
    elif signal in ("POSSIBLE_FAKE", "POSSIBLE_TRUE"):
        base = 0.60 + (top_a - 0.84) * 2.5
        return round(min(0.80, max(0.55, base)), 2)
    elif signal in ("POSSIBLE_MATCH", "POSSIBLE_PARTIAL_MATCH"):
        return 0.55
    elif signal == "EVIDENCE_FOUND":
        if top_rerank is not None:
            return round(min(0.80, max(0.35, (top_rerank + 5) / 12)), 2)
        return 0.45
    else:
        return 0.20

def fact_check_claim(claim: str, k: int = 5, verbose: bool = True, classifier_signal: dict = None) -> dict:
    """
    Cascade retrieval:
      Stage 1 — Bucket A (E5-large cosine vs ALL verified claims, labeled)
                HIGH     ≥ 0.86  → near-identical claim, trust directly, skip Bucket B
                POSSIBLE ≥ 0.84  → topically related, ALSO search Bucket B for more evidence
                < 0.84   → fall through to Bucket B
      Stage 2 — Bucket B (RRF E5+BM25 + NER boosting + cross-encoder rerank)
                Reached when: no Bucket A match (verdict=None) OR POSSIBLE-tier match (gather more evidence)
    Output includes final_verdict (TRUE/FALSE/UNVERIFIED) + confidence (0-1).

    classifier_signal: optional dict matching System 1's output schema
        (label, label_id, confidence, probabilities). Passed through
        unmodified into the output JSON under "classifier_signal" — fusion
        with this system's verdict happens downstream in System 3
        (Youssef), per both schemas' stated design ("to be fused with
        the RAG system's independent verdict"). This system does not
        alter its own verdict logic based on it.
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"CLAIM: {claim}")
        print(f"{'='*60}")

    dialect, conf = detect_dialect(claim)
    query_text    = prepare_query(claim, verbose=verbose)
    query_vec     = get_embedding(query_text, prefix="query")

    # ── Stage 1: Bucket A ────────────────────────────────────────────────────
    results_a = col_a.query(query_embeddings=[query_vec.tolist()], n_results=3)
    bucket_a_hits = [
        {
            "claim":      doc,
            "similarity": round(1 - dist, 3),
            "label":      meta.get("label", "UNKNOWN"),
            "source":     meta.get("source", ""),
            "dialect":    meta.get("dialect", ""),
            "debunk":     meta.get("description", "")[:200],
        }
        for doc, dist, meta in zip(
            results_a['documents'][0],
            results_a['distances'][0],
            results_a['metadatas'][0])
    ]

    # Near-tie disambiguation — generalizes beyond any single case:
    # Chroma ranks by cosine similarity, but a 0.003 gap between #1 and #2
    # is measurement noise, not a meaningful signal that #1 is "more correct."
    # If the top candidates are within NEAR_TIE_MARGIN of each other AND
    # disagree on label (e.g. one FALSE, one TRUE — common when AraFacts has
    # multiple distinct historical claims about the same recurring topic),
    # blindly trusting rank #1 risks an arbitrary, noise-driven verdict.
    # Fix: among near-tied candidates, let lexical overlap with the actual
    # query (word-level grounding) break the tie instead of cosine rank.
    NEAR_TIE_MARGIN = 0.02

    if bucket_a_hits:
        top_score = bucket_a_hits[0]['similarity']
        tied_cluster = [h for h in bucket_a_hits if top_score - h['similarity'] <= NEAR_TIE_MARGIN]
        labels_in_cluster = {h['label'] for h in tied_cluster}

        if len(tied_cluster) > 1 and len(labels_in_cluster) > 1:
            # Conflicting labels among near-tied candidates — use lexical
            # overlap with the query to pick the best-grounded one.
            for h in tied_cluster:
                h['_tiebreak_overlap'] = lexical_overlap(query_text, h['claim'])
            tied_cluster.sort(key=lambda h: h['_tiebreak_overlap'], reverse=True)
            best = tied_cluster[0]

            if verbose:
                print(f"\n[Bucket A — Near-tie conflict detected]")
                for h in tied_cluster:
                    print(f"  sim={h['similarity']:.3f} overlap={h['_tiebreak_overlap']:.3f} [{h['label']}] {h['claim'][:60]}")
                print(f"  → resolved to: [{best['label']}] (highest lexical grounding)")

            # Promote the winning candidate to rank #1 so downstream logic uses it.
            bucket_a_hits.remove(best)
            bucket_a_hits.insert(0, best)

        # Strip the debug-only tiebreak field — internal calculation detail,
        # not part of the documented output schema.
        for h in bucket_a_hits:
            h.pop('_tiebreak_overlap', None)

    top_a     = bucket_a_hits[0]['similarity'] if bucket_a_hits else 0
    top_label = bucket_a_hits[0]['label']      if bucket_a_hits else "UNKNOWN"

    if verbose:
        print(f"\n[Bucket A — Verified Claims DB]")
        for i, h in enumerate(bucket_a_hits[:2], 1):
            print(f"  #{i} ({h['similarity']:.3f}) [{h['label']}] [{h['source']}] {h['claim'][:65]}")

    # ── Verdict signal (Bucket A tier) ───────────────────────────────────────
    if top_a >= 0.86:
        verdict = "HIGH_FAKE_MATCH"    if top_label == "FALSE" else \
                "HIGH_TRUE_MATCH"    if top_label == "TRUE"   else \
                "HIGH_PARTIAL_MATCH"
    elif top_a >= 0.84:
        verdict = "POSSIBLE_FAKE"      if top_label == "FALSE" else \
                "POSSIBLE_TRUE"      if top_label == "TRUE"   else \
                "POSSIBLE_MATCH"
    else:
        verdict = None

    # Lexical sanity check — catch high E5 similarity with near-zero word overlap
    suspicious_match = False
    if verdict is not None and bucket_a_hits:
        overlap = lexical_overlap(query_text, bucket_a_hits[0]['claim'])
        if overlap < 0.20:  # calibrated via F1 sweep on 197 labeled pairs (was 0.15, see reranker_calibration.ipynb)
            suspicious_match = True
            if verbose:
                print(f"  ⚠️  High similarity ({top_a:.3f}) but low lexical overlap ({overlap:.3f}) — distrusting match, forcing Bucket B")

    # Only stop early for HIGH-tier AND not suspicious
    if verdict is not None and verdict.startswith("HIGH") and not suspicious_match:
        final_verdict = VERDICT_TO_FINAL[verdict]
        confidence    = compute_confidence(verdict, top_a=top_a)
        if verbose:
            print(f"\n[Verdict Signal]: {verdict}")
            print(f"[Final Verdict]:  {final_verdict}  (confidence={confidence})")
            print(f"  Bucket B skipped (HIGH confidence)")
        return {
            "claim": claim, "dialect": dialect, "dialect_confidence": conf,
            "query_used": query_text, "verdict_signal": verdict,
            "final_verdict": final_verdict, "confidence": confidence,
            "bucket_a": bucket_a_hits, "bucket_b": [], "bucket_b_searched": False,
            "classifier_signal": classifier_signal,
        }

    # FIXED: explicit mapping instead of blind string replace
    if suspicious_match and verdict in HIGH_TO_POSSIBLE:
        verdict = HIGH_TO_POSSIBLE[verdict]

    # ── Stage 2: Bucket B ────────────────────────────────────────────────────
    if verbose:
        reason = f"POSSIBLE signal ({verdict})" if verdict else f"weak Bucket A ({top_a:.3f})"
        print(f"  {reason} → searching Bucket B…")

    bucket_b_raw  = hybrid_search(query_text, k=20)
    bucket_b_hits = rerank(query_text, bucket_b_raw, top_k=k)
    top_rerank    = bucket_b_hits[0]['rerank_score'] if bucket_b_hits else None

    # Threshold below which Bucket B's "top hit" is noise, not a real candidate —
    # distinct from EVIDENCE_FOUND's >2 cutoff. Below this, Bucket B is silent
    # (no relevant evidence exists, e.g. a KB coverage gap), not contradicting.
    NO_EVIDENCE_RERANK = -2.0

    if verdict is not None:
        final_signal = verdict
        confidence   = compute_confidence(verdict, top_a=top_a)
        # FIX (bug #4): don't blindly trust a carried-over POSSIBLE signal —
        # apply the same lexical-overlap entity-confusion check (crocodile
        # case, Sciavolino et al. 2021) to Bucket B's own top hit before
        # accepting it. Without this, a vague claim like "مشروع دبي القادم"
        # that shares only a place name with the evidence gets confidently
        # mismatched.
        #
        # BUT: only apply this check when Bucket B actually returned a plausible
        # candidate (rerank > NO_EVIDENCE_RERANK). A deeply negative rerank score
        # means Bucket B found nothing relevant at all (KB coverage gap) — that's
        # silence, not contradiction, and silence should not override a
        # reasonably-grounded Bucket A match. Conflating "no evidence" with
        # "contradicting evidence" was downgrading correct Bucket A verdicts
        # whenever the KB simply didn't cover the topic (e.g. metro ticket
        # pricing — Bucket B has metro construction news, zero pricing coverage).
        if bucket_b_hits and top_rerank is not None and top_rerank > NO_EVIDENCE_RERANK:
            b_overlap = lexical_overlap(query_text, bucket_b_hits[0]['proposition'])
            if b_overlap < 0.20:  # calibrated threshold (was 0.15)
                if verbose:
                    print(f"  ⚠️  Bucket B top hit shares almost no words with claim (overlap={b_overlap:.3f}) — downgrading to LOW_CONFIDENCE")
                final_signal = "LOW_CONFIDENCE"
                confidence   = compute_confidence(final_signal, top_rerank=top_rerank)
        elif verbose:
            print(f"  Bucket B found no real candidates (top_rerank={top_rerank}) — sticking with Bucket A verdict ({verdict})")
    else:
        final_signal = "EVIDENCE_FOUND" if (top_rerank is not None and top_rerank > 2) else "LOW_CONFIDENCE"  # calibrated threshold (F1 0.471 -> 0.667)
        confidence   = compute_confidence(final_signal, top_rerank=top_rerank)

    final_verdict = VERDICT_TO_FINAL[final_signal]

    # Attach a decontextualized version of the top evidence for display (cached + fail-safe)
    if bucket_b_hits:
        bucket_b_hits[0]['proposition_display'] = get_decontextualized(
            bucket_b_hits[0]['title'], bucket_b_hits[0]['proposition']
        )

    if verbose:
        print(f"\n[Bucket B — RRF + Rerank]")
        for i, h in enumerate(bucket_b_hits[:3], 1):
            print(f"  #{i} (rerank={h.get('rerank_score','?'):.3f}) {h['proposition'][:70]}")
            print(f"       ↳ {h['title'][:60]}")
        print(f"\n[Verdict Signal]: {final_signal}")
        print(f"[Final Verdict]:  {final_verdict}  (confidence={confidence})")

    return {
        "claim":              claim,
        "dialect":            dialect,
        "dialect_confidence": conf,
        "query_used":         query_text,
        "verdict_signal":     final_signal,
        "final_verdict":      final_verdict,
        "confidence":         confidence,
        "bucket_a":           bucket_a_hits,
        "bucket_b":           bucket_b_hits,
        "bucket_b_searched":  True,
        "classifier_signal":  classifier_signal,
    }

print("✅ fact_check_claim ready (+ cached, fail-safe decontextualization of displayed evidence)")

Quick sanity check — run one claim to confirm everything loaded correctly.

In [ ]:
fact_check_claim("لقاح أسترازينيكا يسبب جلطات دموية")

**Source links** — maps article titles to their original URLs, shown in the demo UI.

In [ ]:
_truth_links = pd.read_csv(f"{BASE}/Bucket_B_Truth_CLEAN_FINAL.csv")
TITLE_TO_LINK = dict(zip(_truth_links['title'], _truth_links['link']))
print(f"✅ Loaded {len(TITLE_TO_LINK)} title->link mappings")

def show_case(claim, k=5):
    r = fact_check_claim(claim, k=k, verbose=False)
    print("="*70)
    print(f"CLAIM:           {r['claim']}")
    print(f"DIALECT:         {r['dialect']} (conf={r['dialect_confidence']:.2f})")
    print(f"QUERY USED:      {r['query_used']}")
    print("-"*70)
    print(f"VERDICT:         {r['final_verdict']}")
    print(f"CONFIDENCE:      {r['confidence']}")
    print(f"SIGNAL:          {r['verdict_signal']}")
    print(f"BUCKET B USED:   {r['bucket_b_searched']}")
    if r.get('retry_triggered'):
        print(f"RETRY FIRED:     yes -> '{r['reformulated_query']}'")
    print("-"*70)
    if r['bucket_a']:
        top = r['bucket_a'][0]
        print(f"BUCKET A TOP MATCH (sim={top['similarity']}):")
        print(f"  [{top['label']}] {top['claim']}")
        print(f"  Source: {top['source']}")
    if r['bucket_b']:
        top = r['bucket_b'][0]
        link = TITLE_TO_LINK.get(top['title'], "(no link found for this title)")
        print(f"\nBUCKET B TOP EVIDENCE (rerank={top['rerank_score']:.3f}):")
        print(f"  {top['proposition']}")
        print(f"  Article title: {top['title']}")
        print(f"  Source: {top['source']}")
        print(f"  Link: {link}")
    print("="*70)

show_case("مشروع دبي القادم")

## 3. Demo interface
Optional — a Gradio web UI for manually testing claims. Not required for System 3 integration.

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

def run_demo(claim):
    if not claim.strip():
        return "<p style='color:#888'>أدخل ادعاءً للتحقق منه</p>"
    
    result = fact_check_claim(claim, verbose=False)
    
    verdict   = result['final_verdict']
    signal    = result['verdict_signal']
    confidence = result['confidence']
    conf_pct  = int(confidence * 100)
    
    color = {"TRUE": "#22c55e", "FALSE": "#ef4444", "UNVERIFIED": "#f59e0b"}.get(verdict, "#888")
    verdict_ar = {"TRUE": "\u0635\u062d\u064a\u062d \u2713", "FALSE": "\u0643\u0627\u0630\u0628 \u2717", "UNVERIFIED": "\u063a\u064a\u0631 \u0645\u0624\u0643\u062f \u26a0"}.get(verdict, verdict)

    html = f"""
    <div style="font-family: system-ui; direction: rtl; text-align: right; max-width: 720px;">
      <div style="background:#1a1d27; border-radius:10px; padding:16px; margin-bottom:14px;">
        <div style="color:#888; font-size:12px; margin-bottom:6px;">{chr(0x1F4DD)} \u0627\u0644\u0627\u062f\u0639\u0627\u0621</div>
        <div style="color:#f0f0f0; font-size:15px;">{result['claim']}</div>
        <div style="color:#555; font-size:11px; margin-top:6px;">
          \u0627\u0644\u0644\u0647\u062c\u0629: {result['dialect']} ({int(result['dialect_confidence']*100)}%) \u00b7 
          \u0627\u0644\u0627\u0633\u062a\u0639\u0644\u0627\u0645: {result['query_used']}
        </div>
      </div>
      <div style="background:#1a1d27; border-radius:10px; padding:20px; margin-bottom:14px; border:2px solid {color};">
        <div style="font-size:32px; font-weight:800; color:{color};">{verdict_ar}</div>
        <div style="color:#888; font-size:13px; margin-top:4px;">{signal}</div>
        <div style="margin-top:12px; background:#111; border-radius:6px; height:10px;">
          <div style="background:{color}; width:{conf_pct}%; height:10px; border-radius:6px;"></div>
        </div>
        <div style="color:#888; font-size:12px; margin-top:4px;">\u0627\u0644\u062b\u0642\u0629: {conf_pct}%</div>
      </div>
    """

    if result['bucket_a']:
        html += """
      <div style="background:#1a1d27; border-radius:10px; padding:16px; margin-bottom:14px; border:1px solid #1d4ed8;">
        <div style="color:#60a5fa; font-size:13px; font-weight:600; margin-bottom:12px;">
          \U0001f5c4\ufe0f \u0645\u0637\u0627\u0628\u0642\u0629 \u0645\u0646 \u0642\u0627\u0639\u062f\u0629 \u0627\u0644\u0627\u062f\u0639\u0627\u0621\u0627\u062a \u0627\u0644\u0645\u0648\u062b\u0642\u0629
        </div>"""
        for h in result['bucket_a'][:2]:
            lc = {"TRUE":"#22c55e","FALSE":"#ef4444","PARTLY_FALSE":"#f59e0b"}.get(h['label'],"#888")
            debunk_text = h.get('debunk', '') or ''
            html += f"""
        <div style="border-top:1px solid #334155; padding:12px 0;">
          <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">
            <span style="color:{lc}; font-size:13px; font-weight:700; background:{lc}22; padding:3px 10px; border-radius:5px;">{h['label']}</span>
            <span style="color:#888; font-size:11px;">\u0627\u0644\u0645\u0635\u062f\u0631: {h['source']} \u00b7 \u062a\u0634\u0627\u0628\u0647: {h['similarity']}</span>
          </div>
          <div style="color:#e2e8f0; font-size:14px; margin-bottom:6px;">{h['claim']}</div>
          {"<div style='background:#111; border-radius:6px; padding:10px; margin-top:6px;'><div style=color:#888;font-size:10px;margin-bottom:4px;>\U0001f4cb \u0627\u0644\u062a\u0641\u0646\u064a\u062f:</div><div style=color:#aaa;font-size:12px;line-height:1.6;>" + debunk_text + "</div></div>" if debunk_text else ""}
        </div>"""
        html += "</div>"

    if result.get('bucket_b_searched') and result.get('bucket_b'):
        html += """
      <div style="background:#1a1d27; border-radius:10px; padding:16px; margin-bottom:14px; border:1px solid #7c3aed;">
        <div style="color:#a78bfa; font-size:13px; font-weight:600; margin-bottom:12px;">
          \U0001f4f0 \u0623\u062f\u0644\u0629 \u0645\u0646 \u0642\u0627\u0639\u062f\u0629 \u0627\u0644\u0645\u0639\u0631\u0641\u0629 \u0627\u0644\u0625\u062e\u0628\u0627\u0631\u064a\u0629
        </div>"""
        for h in result['bucket_b'][:3]:
            display_text = h.get('proposition_display', h['proposition'])
            link = TITLE_TO_LINK.get(h.get('title', ''), '')
            score = h.get('rerank_score', 0)
            score_color = '#22c55e' if score > 5 else '#f59e0b' if score > 2 else '#888'
            html += f"""
        <div style="border-top:1px solid #334155; padding:12px 0;">
          <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
            <span style="color:{score_color}; font-size:11px; font-weight:600;">\u062f\u0631\u062c\u0629 \u0627\u0644\u062a\u0637\u0627\u0628\u0642: {score:.2f}</span>
            <span style="color:#555; font-size:11px;">{h.get('source','')}</span>
          </div>
          <div style="color:#e2e8f0; font-size:13px; line-height:1.6; margin-bottom:6px;">{display_text[:200]}</div>
          <div style="color:#777; font-size:11px;">\U0001f4c4 {h.get('title','')[:100]}</div>
          {"<a href='" + link + "' target='_blank' style='display:inline-block; margin-top:6px; color:#60a5fa; font-size:11px; text-decoration:none; background:#1d4ed811; padding:4px 10px; border-radius:5px;'>\U0001f517 \u0631\u0627\u0628\u0637 \u0627\u0644\u0645\u0635\u062f\u0631 \u0627\u0644\u0623\u0635\u0644\u064a \u2190</a>" if link else "<div style='color:#555; font-size:10px; margin-top:4px;'>\u26a0\ufe0f \u0644\u0627 \u064a\u0648\u062c\u062f \u0631\u0627\u0628\u0637 \u0645\u062a\u0627\u062d</div>"}
        </div>"""
        html += "</div>"

    elif result.get('bucket_b_searched') and not result.get('bucket_b'):
        html += """<div style="background:#1a1d27; border-radius:8px; padding:12px; margin-bottom:14px; border:1px solid #333;">
          <div style="color:#888; font-size:12px;">\U0001f4f0 \u0644\u0645 \u064a\u064f\u0639\u062b\u0631 \u0639\u0644\u0649 \u0623\u062f\u0644\u0629 \u0625\u0636\u0627\u0641\u064a\u0629</div>
        </div>"""

    html += "</div>"
    return html


with gr.Blocks(theme=gr.themes.Base(), title="System 2A \u2014 RAG Fact-Verification") as demo:
    gr.Markdown("# \U0001f50d System 2A \u2014 RAG Fact-Verification Engine\n**A Hybrid AI Framework for Arabic Fake News Detection**\n\nKB: 28,800+ propositions from 2,000+ articles \u00b7 7 live RSS sources \u00b7 auto-updates every 4h")
    
    with gr.Row():
        with gr.Column(scale=2):
            claim_input = gr.Textbox(
                label="\u0623\u062f\u062e\u0644 \u0627\u0644\u0627\u062f\u0639\u0627\u0621 \u0628\u0627\u0644\u0639\u0631\u0628\u064a\u0629",
                placeholder="\u0645\u062b\u0627\u0644: \u0641\u064a\u0631\u0648\u0633 \u0643\u0648\u0631\u0648\u0646\u0627 \u0627\u062a\u0639\u0645\u0644 \u0641\u064a \u0645\u062e\u062a\u0628\u0631",
                lines=3,
                rtl=True,
            )
            submit_btn = gr.Button("\U0001f50d \u062a\u062d\u0642\u0642", variant="primary")
        
    output = gr.HTML()
    
    gr.Examples(
        examples=[
            ["\u0645\u0641\u064a\u0634 \u062f\u0644\u064a\u0644 \u0639\u0644\u0649 \u0627\u0646 \u0641\u064a\u0631\u0648\u0633 \u0643\u0648\u0631\u0648\u0646\u0627 \u0627\u062a\u0639\u0645\u0644 \u0641\u064a \u0645\u062e\u062a\u0628\u0631"],
            ["\u0627\u0644\u0633\u064a\u0633\u064a \u0628\u0627\u0639 \u0633\u064a\u0646\u0627\u0621 \u0644\u0644\u0625\u0633\u0631\u0627\u0626\u064a\u0644\u064a\u064a\u0646"],
            ["\u0623\u0633\u0639\u0627\u0631 \u0627\u0644\u0646\u0641\u0637 \u0627\u0631\u062a\u0641\u0639\u062a \u0628\u0633\u0628\u0628 \u0627\u0644\u062a\u0648\u062a\u0631\u0627\u062a \u0641\u064a \u0645\u0646\u0637\u0642\u0629 \u0627\u0644\u062e\u0644\u064a\u062c"],
            ["\u0644\u0642\u0627\u062d \u0623\u0633\u062a\u0631\u0627\u0632\u064a\u0646\u064a\u0643\u0627 \u064a\u0633\u0628\u0628 \u062c\u0644\u0637\u0627\u062a \u062f\u0645\u0648\u064a\u0629"],
            ["\u0627\u0644\u062c\u0627\u0645\u0639\u0629 \u0627\u0644\u0639\u0631\u0628\u064a\u0629 \u0639\u0642\u062f\u062a \u0627\u062c\u062a\u0645\u0627\u0639\u0627\u064b \u0637\u0627\u0631\u0626\u0627\u064b \u0644\u0628\u062d\u062b \u0627\u0644\u0623\u0632\u0645\u0629 \u0627\u0644\u0633\u0648\u0631\u064a\u0629"],
            ["\u0625\u0633\u0631\u0627\u0626\u064a\u0644 \u062a\u0635\u0639\u062f \u0639\u0645\u0644\u064a\u0627\u062a\u0647\u0627 \u0627\u0644\u0639\u0633\u0643\u0631\u064a\u0629 \u0641\u064a \u0642\u0637\u0627\u0639 \u063a\u0632\u0629"],
        ],
        inputs=claim_input,
    )
    
    submit_btn.click(fn=run_demo, inputs=claim_input, outputs=output)
    claim_input.submit(fn=run_demo, inputs=claim_input, outputs=output)

demo.launch(share=True)
print("\u2705 Demo running \u2014 share link above")


## 4. Output for System 3 (Youssef)
Runs a batch of test claims through this system **and** System 1's real classifier, then saves the combined output to `system2b/batch_test_cases.json` — the file System 3 tests its fusion logic against. See `system2b/output_schema.json` for the field-by-field contract.

In [ ]:
# ── Batch export: multiple test cases for Youssef's integration testing ────
# Uses Sarah's ACTUAL trained classifier (Hugging Face: sarahhh33/arabic-fake-news-marbert),
# not a mock — real System 1 output, fused at the JSON level with this system's
# real output, per the agreed sequential architecture (Sarah -> Russell -> Youssef).

import sys
sys.path.insert(0, "/Users/russelltamer/Desktop/system 1")
from classifier import FakeNewsClassifier

print("Loading Sarah's classifier from Hugging Face...")
sarah_clf = FakeNewsClassifier(model_path="sarahhh33/arabic-fake-news-marbert")
print("✅ Sarah's classifier loaded\n")

batch_test_claims = [
    {
        "claim": "السيسي باع سيناء للإسرائيليين",
        "note": "Expected: HIGH_FAKE_MATCH — near-identical to a known debunked claim."
    },
    {
        "claim": "انتشر زعم حول رفع الحكومة لسعر تذاكر مترو الأنفاق بنسبة ١٠٠٪",
        "note": ("Expected: POSSIBLE_TRUE. Cross-system check case — classifier may show "
                 "framing-phrase bias toward 'real' on 'انتشر/زعم' (System 1's documented "
                 "limitation). RAG verdict is grounded in the actual evidence after the "
                 "near-tie Bucket A fix. Good sanity check for fusion logic.")
    },
    {
        "claim": "لازم اشرب ٣ لتر مياه في اليوم",
        "note": "Expected: EVIDENCE_FOUND/UNVERIFIED — general health claim, no Bucket A match, Bucket B has relevant supporting evidence."
    },
    {
        "claim": "ايه اخبار الجو اليوم",
        "note": "Expected: LOW_CONFIDENCE/UNVERIFIED — not a real factual claim, just a question."
    },
    {
        "claim": "مفيش دليل على ان فيروس كورونا اتعمل في مختبر",
        "note": "Expected: tests dialect handling (Egyptian) + a well-covered topic with both buckets populated."
    },
]

batch_results = []
for case in batch_test_claims:
    classifier_signal = sarah_clf.predict(case["claim"])  # REAL System 1 output
    result = fact_check_claim(case["claim"], classifier_signal=classifier_signal, verbose=False)
    result["_test_note"] = case["note"]
    batch_results.append(result)
    print(f"[{result['final_verdict']:>10}] conf={result['confidence']:.2f}  "
          f"| Sarah: {classifier_signal['label']} ({classifier_signal['confidence']:.2f})  "
          f"{case['claim'][:45]}")

with open(f"{BASE}/system2b/batch_test_cases.json", "w", encoding="utf-8") as f:
    json.dump(batch_results, f, ensure_ascii=False, indent=2)

print(f"\n✅ Saved {len(batch_results)} test cases (real Sarah + real Russell output) to system2b/batch_test_cases.json")


## 5. Appendix — evaluation & calibration (optional, not needed to run the pipeline)
This section contains the evidence behind the design choices above: retrieval quality metrics (P@1/R@3/MRR), dialect robustness tests, and threshold calibration (Bucket A and Bucket B). Useful for the thesis writeup; **skip this section if you just want to run the system.**

In [ ]:
# Evaluation — Retrieval Quality (P@1, R@3, MRR)

Compares three retrieval systems on the same test set:
- **BM25 only**
- **AraBERT only**
- **Hybrid (our system)**

Test set: 200 AraFacts claims with their fact-checking descriptions.
Ground truth: for each claim, its own description is the correct document to retrieve.

In [ ]:

# ── Evaluation Setup ─────────────────────────────────────────────────────────
# We create a temporary eval collection with 200 AraFacts descriptions,
# then query each with its matching claim and check rank of correct doc.

N_EVAL    = 200   # number of test pairs
K_EVAL    = 5     # retrieve top-K
EVAL_SEED = 42

# Sample N_EVAL claims that have descriptions
df_eval = df_arafacts[
    df_arafacts['description'].notna() &
    (df_arafacts['description'].str.len() > 60) &
    (df_arafacts['claim'].str.len() > 20)
].sample(N_EVAL, random_state=EVAL_SEED).reset_index(drop=True)

print(f"Eval set: {len(df_eval)} claim-description pairs")
print(f"Label distribution:\n{df_eval['normalized_label'].value_counts().to_string()}")

# ── Build eval collection (temporary, separate from Bucket B) ────────────────
EVAL_COLLECTION = "eval_temp"
try: client.delete_collection(EVAL_COLLECTION)
except: pass
col_eval = client.create_collection(EVAL_COLLECTION, metadata={"hnsw:space": "cosine"})

print("\nEmbedding eval descriptions...")
eval_descs = df_eval['description'].tolist()
vecs_eval  = get_embeddings_batch(eval_descs, batch_size=32)

col_eval.add(
    ids=[f"eval_{i}" for i in range(len(df_eval))],
    embeddings=vecs_eval.tolist(),
    documents=eval_descs,
    metadatas=[{"claim_id": str(r['ClaimID'])} for _, r in df_eval.iterrows()]
)
print(f"✅ Eval collection: {col_eval.count()} descriptions")

# BM25 index for eval descriptions
tokenized_eval = [bm25_tokenize(d) for d in eval_descs]
bm25_eval      = BM25Okapi(tokenized_eval)
print("✅ BM25 eval index ready")

In [ ]:
def get_rank(retrieved: list, correct) -> int | None:
    """Return 1-based rank of correct item, or None if not found."""
    for i, r in enumerate(retrieved):
        if r == correct:
            return i + 1
    return None

def eval_bm25(query: str, correct_idx: int) -> int | None:
    tokens = bm25_tokenize(query)
    scores = bm25_eval.get_scores(tokens)
    top_k  = np.argsort(scores)[::-1][:K_EVAL].tolist()
    return get_rank(top_k, correct_idx)

def eval_e5(query: str, correct_id: str) -> int | None:
    qvec    = get_embedding(query, prefix="query")
    results = col_eval.query(query_embeddings=[qvec.tolist()], n_results=K_EVAL)
    return get_rank(results['ids'][0], correct_id)

def eval_rrf(query: str, correct_idx: int, correct_id: str, k_rrf: int = 60) -> int | None:
    # BM25 ranking
    tokens     = bm25_tokenize(query)
    bm25_sc    = bm25_eval.get_scores(tokens)
    bm25_order = np.argsort(bm25_sc)[::-1]
    bm25_ranks = {int(idx): rank + 1 for rank, idx in enumerate(bm25_order)}

    # E5 ranking
    qvec    = get_embedding(query, prefix="query")
    results = col_eval.query(query_embeddings=[qvec.tolist()], n_results=len(eval_descs))
    e5_ranks = {int(rid.split('_')[1]): rank + 1
                for rank, rid in enumerate(results['ids'][0])}

    # RRF fusion
    all_ids = set(bm25_ranks) | set(e5_ranks)
    rrf_scores = {}
    for idx in all_ids:
        r_bm25 = bm25_ranks.get(idx, len(eval_descs))
        r_e5   = e5_ranks.get(idx, len(eval_descs))
        rrf_scores[idx] = 1 / (k_rrf + r_bm25) + 1 / (k_rrf + r_e5)

    top_k = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)[:K_EVAL]
    return get_rank(top_k, correct_idx)

def compute_metrics(ranks: list) -> dict:
    n = len(ranks)
    return {
        "P@1": round(sum(1 for r in ranks if r == 1) / n, 3),
        "R@3": round(sum(1 for r in ranks if r and r <= 3) / n, 3),
        "MRR": round(sum(1/r for r in ranks if r) / n, 3),
    }

# ── Run ───────────────────────────────────────────────────────────────────────
print(f"Evaluating {len(df_eval)} test pairs  (K={K_EVAL})...\n")

ranks_bm25, ranks_e5, ranks_rrf = [], [], []

for i, row in df_eval.iterrows():
    q   = row['claim']
    ci  = i
    cid = f"eval_{i}"
    ranks_bm25.append(eval_bm25(q, ci))
    ranks_e5.append(eval_e5(q, cid))
    ranks_rrf.append(eval_rrf(q, ci, cid))
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(df_eval)} done...")

# ── Results ───────────────────────────────────────────────────────────────────
mb   = compute_metrics(ranks_bm25)
me   = compute_metrics(ranks_e5)
mrrf = compute_metrics(ranks_rrf)

# v1 results (AraBERT + weighted sum alpha=0.65) — from previous run
v1_bm25    = {"P@1": 0.880, "R@3": 0.920, "MRR": 0.899}
v1_arabert = {"P@1": 0.415, "R@3": 0.630, "MRR": 0.527}
v1_hybrid  = {"P@1": 0.885, "R@3": 0.930, "MRR": 0.909}

print("\n" + "="*60)
print(f"{'System':<28} {'P@1':>6} {'R@3':>6} {'MRR':>6}")
print("-"*60)
print(f"{'── v1 (AraBERT + weighted sum) ──'}")
print(f"  {'BM25 only':<26} {v1_bm25['P@1']:>6} {v1_bm25['R@3']:>6} {v1_bm25['MRR']:>6}")
print(f"  {'AraBERT only':<26} {v1_arabert['P@1']:>6} {v1_arabert['R@3']:>6} {v1_arabert['MRR']:>6}")
print(f"  {'Hybrid v1':<26} {v1_hybrid['P@1']:>6} {v1_hybrid['R@3']:>6} {v1_hybrid['MRR']:>6}")
print(f"{'── v2 (E5-large + RRF) ──'}")
print(f"  {'BM25 only':<26} {mb['P@1']:>6} {mb['R@3']:>6} {mb['MRR']:>6}")
print(f"  {'E5-large only':<26} {me['P@1']:>6} {me['R@3']:>6} {me['MRR']:>6}")
print(f"  {'RRF Hybrid v2 (ours)':<26} {mrrf['P@1']:>6} {mrrf['R@3']:>6} {mrrf['MRR']:>6}")
print("="*60)
print(f"\nTest set: {len(df_eval)} pairs  |  K={K_EVAL}  |  same seed as v1")

# Cleanup
try: client.delete_collection("eval_temp")
except: pass
print("✅ Eval complete")

In [ ]:
# ── Dialect Robustness Evaluation ────────────────────────────────────────────
# Proves hybrid + normalization outperforms BM25 alone on Egyptian dialect input.
# Strategy: inject EGY dialect words into MSA eval queries, then compare:
#   - BM25 raw (no normalization) → degrades on dialect
#   - Hybrid + normalize_to_msa  → stays strong

# MSA → EGY injection map (reverse of EGY_TO_MSA subset)
MSA_TO_EGY_INJECT = {
    "لا يوجد":  "مفيش",
    "ليس":      "مش",
    "هذا":      "ده",
    "هذه":      "دي",
    "الذي":     "اللي",
    "لأن":      "عشان",
    "لكن":      "بس",
    "أيضاً":    "كمان",
    "الآن":     "دلوقتي",
    "جداً":     "اوي",
    "ذهب":      "راح",
    "جاء":      "جه",
    "رأى":      "شاف",
}

def inject_egy_dialect(text: str) -> str:
    for msa, egy in MSA_TO_EGY_INJECT.items():
        text = text.replace(msa, egy)
    return text

# Rebuild eval collection (was deleted at end of previous cell)
print("Rebuilding eval collection...")
try: client.delete_collection("eval_temp")
except: pass
col_eval = client.create_collection("eval_temp", metadata={"hnsw:space": "cosine"})
eval_descs     = df_eval['description'].tolist()
vecs_eval      = get_embeddings_batch(eval_descs, batch_size=32)
col_eval.add(
    ids=[f"eval_{i}" for i in range(len(df_eval))],
    embeddings=vecs_eval.tolist(),
    documents=eval_descs,
    metadatas=[{"claim_id": str(r['ClaimID'])} for _, r in df_eval.iterrows()]
)
tokenized_eval = [bm25_tokenize(d) for d in eval_descs]
bm25_eval      = BM25Okapi(tokenized_eval)
print(f"✅ Eval collection ready: {col_eval.count()} descriptions")

# Create dialectal versions of eval queries
dialect_queries = [inject_egy_dialect(q) for q in df_eval['claim'].tolist()]
modified = sum(1 for o, d in zip(df_eval['claim'].tolist(), dialect_queries) if o != d)
print(f"Claims modified with EGY dialect injection: {modified}/200\n")

# Run evaluation
print("Running dialect robustness evaluation...")
ranks_bm25_egy, ranks_rrf_egy = [], []

for i, (row, dialect_q) in enumerate(zip(df_eval.itertuples(), dialect_queries)):
    ci  = i
    cid = f"eval_{i}"
    # BM25 on raw dialect (no normalization)
    ranks_bm25_egy.append(eval_bm25(dialect_q, ci))
    # Hybrid with MSA normalization applied first
    normalized_q = normalize_to_msa(dialect_q)
    ranks_rrf_egy.append(eval_rrf(normalized_q, ci, cid))
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/200 done...")

mb_egy = compute_metrics(ranks_bm25_egy)
mh_egy = compute_metrics(ranks_rrf_egy)

print("\n" + "="*60)
print("STANDARD (MSA) vs DIALECT (EGY) — Robustness Comparison")
print("="*60)
print(f"{'System':<35} {'P@1':>6} {'R@3':>6} {'MRR':>6}")
print("-"*60)
print(f"{'BM25 — MSA input':<35} {mb['P@1']:>6} {mb['R@3']:>6} {mb['MRR']:>6}")
print(f"{'BM25 — EGY dialect (no norm)':<35} {mb_egy['P@1']:>6} {mb_egy['R@3']:>6} {mb_egy['MRR']:>6}")
print("-"*60)
print(f"{'Hybrid — MSA input':<35} {mrrf['P@1']:>6} {mrrf['R@3']:>6} {mrrf['MRR']:>6}")
print(f"{'Hybrid + norm — EGY dialect':<35} {mh_egy['P@1']:>6} {mh_egy['R@3']:>6} {mh_egy['MRR']:>6}")
print("="*60)

print(f"\nDialect robustness drop:")
print(f"  BM25:   P@1 {mb['P@1']} → {mb_egy['P@1']}  (Δ {mb_egy['P@1']-mb['P@1']:+.3f})")
print(f"  Hybrid: P@1 {mrrf['P@1']} → {mh_egy['P@1']}  (Δ {mh_egy['P@1']-mrrf['P@1']:+.3f})")
print("\n→ Smaller drop = better dialect robustness")

try: client.delete_collection("eval_temp")
except: pass

In [ ]:
# ── Failure Recovery Analysis ─────────────────────────────────────────────────
# Shows WHERE hybrid beats BM25: specific cases BM25 fails but hybrid recovers.
# Uses ranks_bm25 and ranks_rrf already computed in the evaluation cell.

# Cases where BM25 fails at rank 1 but Hybrid gets rank 1
bm25_fail_hybrid_win = [
    i for i in range(len(ranks_bm25))
    if ranks_bm25[i] != 1 and ranks_rrf[i] == 1
]

# Cases where BM25 wins rank 1 but Hybrid fails
bm25_win_hybrid_fail = [
    i for i in range(len(ranks_bm25))
    if ranks_bm25[i] == 1 and ranks_rrf[i] != 1
]

# Cases where both fail
both_fail = [
    i for i in range(len(ranks_bm25))
    if ranks_bm25[i] != 1 and ranks_rrf[i] != 1
]

print("=" * 60)
print("FAILURE RECOVERY ANALYSIS  (200 test pairs)")
print("=" * 60)
print(f"BM25 fails → Hybrid recovers : {len(bm25_fail_hybrid_win)} cases")
print(f"Hybrid fails → BM25 recovers : {len(bm25_win_hybrid_fail)} cases")
print(f"Both fail                     : {len(both_fail)} cases")
print(f"Both succeed                  : {200 - len(bm25_fail_hybrid_win) - len(bm25_win_hybrid_fail) - len(both_fail)} cases")

# Show example claims where hybrid recovers BM25 failure
print(f"\n── Claims BM25 missed, Hybrid found (first 5) ──")
for i in bm25_fail_hybrid_win[:5]:
    row = df_eval.iloc[i]
    print(f"\n  [{row['normalized_label']}] {row['claim'][:90]}")
    print(f"   BM25 rank: {ranks_bm25[i]}  |  Hybrid rank: {ranks_rrf[i]}")

# Key metric: net gain
net_gain = len(bm25_fail_hybrid_win) - len(bm25_win_hybrid_fail)
print(f"\n── Net gain from Hybrid over BM25: +{net_gain} correctly ranked claims ──")
print(f"   Hybrid uniquely recovers {len(bm25_fail_hybrid_win)} claims BM25 cannot find")
print(f"   Hybrid loses only {len(bm25_win_hybrid_fail)} claims BM25 finds")

# Show avg rank improvement for Hybrid vs BM25 on cases where BM25 does NOT get rank 1
non_trivial = [i for i in range(len(ranks_bm25)) if ranks_bm25[i] != 1]
avg_rank_bm25 = sum(ranks_bm25[i] if ranks_bm25[i] else 6 for i in non_trivial) / len(non_trivial)
avg_rank_rrf  = sum(ranks_rrf[i]  if ranks_rrf[i]  else 6 for i in non_trivial) / len(non_trivial)
print(f"\n── On the {len(non_trivial)} hard cases (BM25 rank ≠ 1) ──")
print(f"   BM25  avg rank: {avg_rank_bm25:.2f}")
print(f"   Hybrid avg rank: {avg_rank_rrf:.2f}")
print(f"   → Hybrid ranks correct answers higher on hard cases")

In [ ]:
# Normalization Ablation — does EGY→MSA normalization improve matching?
test_claims = [
    "مفيش دليل على ان فيروس كورونا اتعمل في مختبر",
    "السيسي باع سيناء للإسرائيليين",
    "مش صحيح ان اللقاح بيخلي الناس تموت",
    "الحكومة بتكدب على الناس في موضوع الكهرباء",
    "مفيش فيروس اسمه كورونا ده كله كدب",
]

print(f"{'Claim':<45} {'Raw':>6} {'Normalized':>12} {'Gain':>6}")
print("-" * 75)
for claim in test_claims:
    # Without normalization
    raw_vec  = get_embedding(claim)
    res_raw  = col_a.query(query_embeddings=[raw_vec.tolist()], n_results=1)
    score_raw = round(1 - res_raw['distances'][0][0], 3)

    # With normalization
    norm_claim = normalize_to_msa(claim)
    norm_vec   = get_embedding(norm_claim)
    res_norm   = col_a.query(query_embeddings=[norm_vec.tolist()], n_results=1)
    score_norm = round(1 - res_norm['distances'][0][0], 3)

    gain = round(score_norm - score_raw, 3)
    print(f"{claim[:44]:<45} {score_raw:>6} {score_norm:>12} {gain:>+6}")

In [ ]:
smoke_test_cases = [
    # (claim, expected_verdict, what_it_checks)
    ("الكذب المغربي يتواصل وسائل إعلام مغربية وصفحات نخ", "UNVERIFIED", "low-overlap Bucket B (should stay UNVERIFIED, bug #4 path)"),
    ("مشروع دبي القادم", "UNVERIFIED", "bug #4 fix — Dubai entity confusion"),
    ("مفيش حاجة اسمها تغير مناخي ده كله كدب", None, "dialect normalization — check query_used for شيء/كذب, no crash"),
    ("الموضوع هذا مهم", None, "dialect — non-final demonstrative reorder"),
    ("عاجل وفاة الفنان المصري المعروف اليوم", None, "generic claim, no Bucket A/B coverage expected"),
]

print(f"{'Result':<8}{'Predicted':<14}{'Expected':<14}Claim / Note")
for claim, expected, note in smoke_test_cases:
    r = fact_check_claim(claim, verbose=False)
    ok = "✅" if (expected is None or r['final_verdict'] == expected) else "❌"
    print(f"{ok:<8}{r['final_verdict']:<14}{str(expected):<14}{note}")
    print(f"        query_used: {r['query_used']}")
    print(f"        signal: {r['verdict_signal']}  bucket_b_searched: {r['bucket_b_searched']}")
    print()

In [ ]:
# ── Bucket A Threshold Calibration (leave-one-out) ──────────────────────────
# Reuses the same 200-claim AraFacts eval set as the P@1 retrieval eval above
# (df_eval, built earlier). Self-matches are explicitly excluded (leave-one-out)
# to avoid the leakage problem documented in Reranker_Calibration_Report.md
# Attempt 3: Bucket A is built FROM AraFacts, so a claim trivially matches
# itself at ~1.0 cosine similarity — that tells us nothing about the threshold.
# Instead we look at each claim's *next-best* match already in the index.

K_LOO = 6  # retrieve a few extra so dropping the self-match still leaves candidates

loo_rows = []
for _, row in df_eval.iterrows():
    claim      = row['claim']
    # BUG FIX: Bucket A's metadata stores labels mapped through LABEL_MAP
    # (e.g. "Partly-false" -> "PARTLY_FALSE", "Sarcasm" -> "SARCASM").
    # Comparing against the raw normalized_label string silently failed
    # for every non-TRUE/FALSE label, making accuracy look like it falls
    # as similarity rises. Apply the same mapping to the ground truth here.
    true_label = LABEL_MAP.get(row['normalized_label'], row['normalized_label'])

    qvec = get_embedding(claim, prefix="query")
    res  = col_a.query(query_embeddings=[qvec.tolist()], n_results=K_LOO)

    candidates = [
        (doc, round(1 - dist, 3), meta)
        for doc, dist, meta in zip(res['documents'][0], res['distances'][0], res['metadatas'][0])
        if doc != claim  # drop exact self-match
    ]
    if not candidates:
        continue

    _, top_sim, top_meta = candidates[0]
    predicted_label = top_meta.get('label', 'UNKNOWN')

    loo_rows.append({
        "claim":           claim,
        "true_label":      true_label,
        "top_sim":         top_sim,
        "predicted_label": predicted_label,
        "correct":         predicted_label == true_label,
    })

df_loo = pd.DataFrame(loo_rows)
print(f"Leave-one-out pairs: {len(df_loo)}")
print(df_loo[['top_sim', 'correct']].describe())

# ── Threshold sweep (same method as Bucket B's rerank_score calibration) ────
thresholds = [0.75, 0.78, 0.80, 0.82, 0.84, 0.86, 0.88, 0.90]
total_correct_possible = df_loo['correct'].sum()

print(f"\n{'Threshold':>10} | {'N triggered':>11} | {'Accuracy':>9} | {'Recall':>7} | {'F1':>6}")
for t in thresholds:
    triggered = df_loo[df_loo['top_sim'] >= t]
    n = len(triggered)
    if n == 0:
        print(f"{t:>10} | {0:>11} | {'-':>9} | {'-':>7} | {'-':>6}")
        continue
    correct  = triggered['correct'].sum()
    accuracy = correct / n                                   # precision: of triggered, how many right
    recall   = correct / total_correct_possible if total_correct_possible else 0
    f1       = 2 * accuracy * recall / (accuracy + recall) if (accuracy + recall) else 0
    print(f"{t:>10} | {n:>11} | {accuracy:>9.3f} | {recall:>7.3f} | {f1:>6.3f}")

print("\nCurrent thresholds in fact_check_claim(): HIGH >= 0.86, POSSIBLE >= 0.84")
print("Compare F1 at 0.86/0.84 against neighboring values above to confirm or revise.")


In [ ]:
# ── Bucket A Threshold Validation — True Near-Duplicate Test (paraphrases) ──
# The leave-one-out test above measures something different from what HIGH/POSSIBLE
# tiers are actually for: it checks if a RANDOM claim's nearest DIFFERENT claim
# shares its label (no reason it should). The real use case is a near-duplicate/
# reworded restatement of an ALREADY fact-checked claim. para_data.json (built
# earlier) has exactly that: 50 original AraFacts claims, each with light/medium/
# heavy/dialect paraphrases — genuine near-duplicates, and leakage-free since the
# paraphrase text itself was never indexed in Bucket A.

with open(f"{BASE}/para_data.json", encoding="utf-8") as f:
    para_data = json.load(f)

para_rows = []
for entry in para_data:
    original = entry["original"]
    match = df_arafacts[df_arafacts["claim"] == original]
    if match.empty:
        continue
    true_label = LABEL_MAP.get(match.iloc[0]["normalized_label"], match.iloc[0]["normalized_label"])

    for level, para_text in entry["paraphrases"].items():
        qvec = get_embedding(para_text, prefix="query")
        res  = col_a.query(query_embeddings=[qvec.tolist()], n_results=1)
        top_sim   = round(1 - res["distances"][0][0], 3)
        predicted = res["metadatas"][0][0].get("label", "UNKNOWN")

        para_rows.append({
            "level":           level,
            "original":        original[:50],
            "top_sim":         top_sim,
            "true_label":      true_label,
            "predicted_label": predicted,
            "correct":         predicted == true_label,
            "would_be_HIGH":     top_sim >= 0.86,
            "would_be_POSSIBLE": 0.84 <= top_sim < 0.86,
        })

df_para_eval = pd.DataFrame(para_rows)
print(f"Paraphrase pairs tested: {len(df_para_eval)}\n")

print("── By paraphrase intensity ──")
print(df_para_eval.groupby("level").agg(
    mean_sim=("top_sim", "mean"),
    accuracy=("correct", "mean"),
    pct_HIGH=("would_be_HIGH", "mean"),
    pct_POSSIBLE=("would_be_POSSIBLE", "mean"),
).round(3))

print("\n── Overall: would current thresholds catch these as near-duplicates? ──")
high_rate     = df_para_eval["would_be_HIGH"].mean()
possible_rate = df_para_eval["would_be_POSSIBLE"].mean()
missed_rate   = (~df_para_eval["would_be_HIGH"] & ~df_para_eval["would_be_POSSIBLE"]).mean()
acc_when_high = df_para_eval[df_para_eval["would_be_HIGH"]]["correct"].mean()

print(f"HIGH tier (>=0.86) hit rate:     {high_rate:.1%}")
print(f"POSSIBLE tier (0.84-0.86) rate:  {possible_rate:.1%}")
print(f"Missed entirely (<0.84) rate:    {missed_rate:.1%}")
print(f"\nAccuracy when HIGH tier triggers: {acc_when_high:.3f}")
